# RGB (VisDrone DET) - 01 Explorar dataset

Que hace: descarga/descomprime VisDrone DET, indexa imagenes y labels, y muestra estadisticas y graficos de exploracion.

Que NO hace: no carga ni entrena ningun modelo (eso es `02_entrenar`).

## Parametros

In [ ]:
DATASET_ZIP = "MyDrive/deteccion_drones_v1/archive.zip"
DATASET_DIR = "visdrone"                        # subcarpeta en env.DATA_ROOT
DATA_YAML   = "VisDrone_Dataset/visdrone.yaml"  # relativo a DATASET_DIR


## Setup

Clona/actualiza el repo, instala el paquete y monta Drive. Identica en todos los notebooks (salvo el import: 01 y 00 no necesitan `experiment.py`).

In [ ]:
REPO = "https://github.com/sbstn-sss/deteccion_drones.git"
REPO_DIR = "/content/deteccion_drones"
BRANCH = "main"   # cambiar para probar una rama antes del merge, ej "feat/framework"

import sys
if "google.colab" in sys.modules:
    !git -C {REPO_DIR} checkout -q {BRANCH} 2>/dev/null && git -C {REPO_DIR} pull -q 2>/dev/null || git clone -q -b {BRANCH} {REPO} {REPO_DIR}
    !pip install -q -e {REPO_DIR}
    sys.path.insert(0, f"{REPO_DIR}/src")   # pip -e usa un .pth que el kernel ya iniciado no lee: sin esto falla el import

%load_ext autoreload
%autoreload 2
from deteccion import env, data, viz
env.mount_drive()


## 1. Obtener el dataset

`unzip_once` no vuelve a descomprimir si `DATASET_DIR` ya tiene contenido (idempotente: Colab es efimero, esta celda se corre en cada sesion nueva).

In [ ]:
import pandas as pd

dataset_dir = env.DATA_ROOT / DATASET_DIR
env.unzip_once(env.drive_path(DATASET_ZIP), dataset_dir)
data_yaml = env.fix_data_yaml(dataset_dir / DATA_YAML)


## 2. Indexar imagenes y labels

Solo rutas y metadatos, no se leen imagenes todavia (dataset puede ser grande).

In [ ]:
d = data.load_data_yaml(data_yaml)
splits = [s for s in ("train", "val", "test") if d["splits"][s] is not None]
images = pd.concat([data.index_split(d, s) for s in splits], ignore_index=True)
boxes, issues = data.load_boxes(images, d["names"])


## 3. Resumen y lineas invalidas

`issues` son lineas de label que no se pudieron interpretar (no se descartan en silencio). Si hay muchas, revisar `reason` antes de entrenar.

In [ ]:
data.summary(images, boxes, issues)


In [ ]:
issues.head()


## 4. Distribucion de clases

In [ ]:
data.class_counts(boxes)


In [ ]:
viz.plot_class_counts(boxes, d["names"])


## 5. Tamanos de imagen y de caja

`sample=2000` evita abrir todas las imagenes si el dataset es grande (solo se lee el header, no el contenido).

In [ ]:
images = data.add_image_sizes(images, sample=2000, seed=0)
viz.plot_image_sizes(images)


In [ ]:
boxes_px = data.add_pixel_sizes(boxes, images)
viz.plot_box_sizes(boxes_px)


## 6. Cajas por imagen

In [ ]:
viz.plot_boxes_per_image(boxes, images)


## 7. Muestras con ground truth

In [ ]:
viz.show_samples(images, boxes, d["names"], n=6, seed=0)


## Conclusiones

_(completar tras revisar los graficos, ej. % de objetos small -> justifica imgsz)_